In [2]:
import pandas as pd

df1 = pd.read_csv('/home/dcm/240222Pat/16S_SILVA138_2_k2db/mpa_output/filtered_combined_bracken_genus_mpa.txt', sep ='\t')
df2 = pd.read_csv('/home/dcm/240812Pat/16S_SILVA138_2_k2db/mpa_output/filtered_combined_bracken_genus_mpa.txt', sep ='\t')

#240222Pat/kraken2_silva_138_2_7_26_25/mpa_output/filtered_combined_bracken_genus_mpa.txt

dfm = pd.merge(df1,df2, on = "#Classification", how = 'outer')
dfm = dfm.fillna(0)

# get read sum across all samples per taxa and keep only taxa that have >0 reads
dfm = dfm[dfm.iloc[:, 1:].sum(axis=1) > 0]

dfm.to_csv('16S_240222Pat_240812Pat/otu_table.txt' , sep = '\t', index = False)

dfm['#Classification'].to_csv('16S_240222Pat_240812Pat/tax.txt' , sep = '\t', index = False)

In [3]:
import pandas as pd

# Example classification data

df = pd.read_csv('16S_240222Pat_240812Pat/tax.txt')
def parse_taxonomy_string(classification):
    parts = classification.split('|')
    result = {}
    for part in parts:
        level, name = part.split('__', 1)
        result[level] = name
    return result

# Apply to the column and expand into new columns
taxonomy_df = df['#Classification'].apply(parse_taxonomy_string).apply(pd.Series)

# Optional: Rename columns to human-readable taxonomy level names
taxonomy_df = taxonomy_df.rename(columns={
    'k': 'Kingdom',
    'p': 'Phylum',
    'c': 'Class',
    'o': 'Order',
    'f': 'Family',
    'g': 'Genus',
    's': 'Species'  # In case you have species info
})

# Merge with original DataFrame (optional)
df = pd.concat([df, taxonomy_df], axis=1)

df.to_csv('16S_240222Pat_240812Pat/tax_table.txt', sep='\t', index=False)


In [4]:
##!pip install taxopy[fuzzy-matching]
import os

import taxopy

import pandas as pd

##!pip install pivottablejs
#from pivottablejs import pivot_ui

from rapidfuzz import process

import seaborn as sns

In [5]:
#wget ftp://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz
taxdb = taxopy.TaxDb(nodes_dmp="/home/dcm/250513Pat_250728Pat_250930Pat/taxdump/nodes.dmp", names_dmp="/home/dcm/250513Pat_250728Pat_250930Pat/taxdump/names.dmp")

In [6]:
#read culture_tax df and clean
df_16s_tax = pd.read_csv('16S_240222Pat_240812Pat/tax_table.txt', sep='\t')
#if isolate has multiple species IDs possible, assume it is a mixed isolate and separate each species ID all as new row
df_16s_tax = df_16s_tax.assign(Genus=df_16s_tax['Genus'].str.split(' / ')).explode('Genus')
df_16s_tax= df_16s_tax[['Genus']]
df_16s_tax = df_16s_tax.drop_duplicates()
df_16s_tax


#function to compare tax name found by taxopy versus input name and pick best match
def best_match(target, choices_dict):
    values = list(choices_dict.values())
    match_value, score, _ = process.extractOne(target, values)

    for key, value in choices_dict.items():
        if value == match_value:
            return {key: (match_value, score)}

    return {}  # return empty dict if no match

#loop over species names, get taxid and matching species name from taxopy
#then get closest match to query
#then get full tax lineage
#append results to output df

df = pd.DataFrame(columns=['Genus','best_taxid','best_match_score','phylum','class','order','family','genus'])

for Genus in df_16s_tax['Genus']:

    try:
        taxids = taxopy.taxid_from_name(f"{Genus}", taxdb, fuzzy=True, score_cutoff=0.80) #set threshold lower to capture missed taxa
        tax_dict = {}

        for taxid in taxids:
            tax_name = taxopy.Taxon(taxid, taxdb).name
            tax_dict[taxid] = tax_name

        best_tax_match = best_match(f"{Genus}", tax_dict)
        best_taxid, best_taxname_score = next(iter(best_tax_match.items()))
        best_taxname = best_taxname_score[0]
        best_match_score = best_taxname_score[1]

        taxa_dict = taxopy.Taxon(best_taxid, taxdb).rank_name_dictionary
        #Append only the selected items
        selected_data = {key: taxa_dict.get(key, 'N/A') for key in ['Genus','best_taxid','best_match_score','phylum','class','order','family','genus']}
        selected_data['Genus'] = f"{Genus}"
        selected_data['best_taxid'] = best_taxid
        selected_data['best_match_score'] = best_match_score
        df = df._append(selected_data, ignore_index=True)
        
    except Exception as e:
        print(f"An error occurred for Genus '{Genus}': {e}")
        continue

In [7]:
df.to_csv('16S_240222Pat_240812Pat/16s_taxids.txt', sep = '\t', index = False)

In [8]:
df_otu = pd.read_csv('16S_240222Pat_240812Pat/otu_table.txt', sep='\t')
df_otu = pd.melt(df_otu, id_vars = "#Classification", var_name = "file", value_name = "read_count")
df_otu = df_otu[df_otu['read_count'] != 0] 
df_otu = df_otu[df_otu['read_count'].notna() & (df_otu['read_count'] != 0)]
df_otu

,#Classification,file,read_count
35,k__Bacillati|p__Actinomycetota|c__Actinomycete...,240222Pat_D24-2534_bracken_genuses.kreport2,39.0
37,k__Bacillati|p__Actinomycetota|c__Actinomycete...,240222Pat_D24-2534_bracken_genuses.kreport2,7.0
38,k__Bacillati|p__Actinomycetota|c__Actinomycete...,240222Pat_D24-2534_bracken_genuses.kreport2,100.0
40,k__Bacillati|p__Actinomycetota|c__Actinomycete...,240222Pat_D24-2534_bracken_genuses.kreport2,227.0
42,k__Bacillati|p__Actinomycetota|c__Actinomycete...,240222Pat_D24-2534_bracken_genuses.kreport2,1473.0
...,...,...,...
35351,k__Bacillati|p__Mycoplasmatota|c__Mollicutes|o...,240812Pat_D24-12952_bracken_genuses.kreport2,2339.0
35367,k__Bacillati|p__Bacillota|c__Clostridia|o__Eub...,240812Pat_D24-12952_bracken_genuses.kreport2,7.0
35368,k__Bacillati|p__Bacillota|c__Clostridia|o__Eub...,240812Pat_D24-12952_bracken_genuses.kreport2,46.0
35371,k__Bacillati|p__Bacillota|c__Negativicutes|o__...,240812Pat_D24-12952_bracken_genuses.kreport2,9.0


In [9]:
df = pd.read_csv('16S_240222Pat_240812Pat/16s_taxids.txt', sep = '\t')
df_tax = pd.read_csv('16S_240222Pat_240812Pat/tax_table.txt', sep='\t')
dfm = pd.merge(df_tax, df, on = 'Genus', how = 'outer')
dfm = pd.merge(dfm, df_otu, on = '#Classification', how = 'outer')
dfm = dfm[dfm['read_count'].notna() & (dfm['read_count'] != 0)]

dfm.to_csv('16S_240222Pat_240812Pat/16s_OTU_taxids.txt', sep = '\t', index = False)